# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [6]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [ ]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 51.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 89.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 95.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.7 MB/s eta 0:00:00
ERROR: pip's dependency resolv

In [8]:
# ── CELL 3: Clone project from GitHub ──────────────────────
!git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git urdu-pipeline
%cd urdu-pipeline

# Pull latest changes
!git pull origin main

print('✔ Repository cloned and updated successfully!')
!ls -la

Cloning into 'urdu-pipeline'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 68 (delta 33), reused 56 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 266.14 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/urdu-pipeline
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
Already up to date.
✔ Repository cloned and updated successfully!
total 44
drwxr-xr-x 5 root root 4096 Jul  1 18:03 .
drwxr-xr-x 1 root root 4096 Jul  1 18:03 ..
-rw-r--r-- 1 root root 3861 Jul  1 18:03 config.py
drwxr-xr-x 8 root root 4096 Jul  1 18:03 .git
-rw-r--r-- 1 root root 9415 Jul  1 18:03 main.py
drwxr-xr-x 2 root root 4096 Jul  1 18:03 notebooks
drwxr-xr-x 2 root root 4096 Jul  1 18:03 pipeline
-rw-r--r-- 1 root root 3756 Jul  1 18:03 README.md
-rw-r--r-- 1 root root 1053 Jul  1 18:03 requirements.txt


In [9]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os

os.makedirs('audio', exist_ok=True)

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

# Trim from 137 seconds, duration 55 minutes (3163 seconds)
!ffmpeg -y -i audio/full.mp3 -ss 137 -t 720 -c copy audio/test_audio.mp3

audio_path = "audio/test_audio.mp3"

size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')
print(f'   Duration: 12 minutes (starting from 137 seconds)')


[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] Destination: audio/full.webm
[download] 100% of   63.59MiB in 00:00:01 at 32.53MiB/s;33m00:000m
[ExtractAudio] Destination: audio/full.mp3
Deleting original file audio/full.webm (pass -k to keep)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfrib

In [10]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.75
  Chunk size       : 500 chars


In [11]:
# Check if audio file exists and its size
import os

audio_path = "audio/test_audio.mp3"
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    size_mins = size_mb / 0.64  # Rough conversion: 1 min ≈ 0.64 MB at 128kbps
    print(f'✔ Audio file exists: {size_mb:.1f} MB ({size_mins:.0f} minutes)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')

# Check GPU status
import torch
if torch.cuda.is_available():
    print(f'✔ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('✗ GPU not available!')

✔ Audio file exists: 6.8 MB (11 minutes)
✔ GPU: Tesla T4
   VRAM: 15.6 GB
   Used: 0.00 GB


In [12]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes for long audio)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.88 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.460 | conf=0.95 | and taking out the time...
    [⚠] 00:00:07.840 → 00:00:10.820 | conf=0.55 | If you give us a small introduction...
    [⚠] 00:00:10.820 → 00:00:13.180 | conf=0.65 | so that our audience that you are listening to today...
    [✔] 00:00:13.180 → 00:00:15.340 | conf=0.82 | they know that what a brilliant...
    [✔] 00:00:15.340 → 00:00:17.200 | conf=0.91 | Academian you are...
    [✔] 00:00:17.200 → 00:00:19.280 | conf=0.93 | Thank you so much Zainab...
    [✔] 00:00:19.280 → 00:00:21.740 | conf=0.88 | for inviting me on your show...
    [✔] 00:00:21.740 → 00:00:24.340 | conf=0.87 | My name is Maj

In [1]:
#Clearing memory of previous model to free up GPU VRAM for next stage
import torch, gc
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

In [14]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF URDU TRANSCRIPT
  Interview ID      : test_audio
  Total segments    : 545
  Confidence threshold: 0.75
    [⚠ FLAGGED] seg 001 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠ FLAGGED] seg 002 | conf=0.72 | Welcome to the show...
    [✔] seg 003 | conf=0.88 | Thank you so much for coming today...
    [✔] seg 004 | conf=0.95 | and taking out the time...
    [⚠ FLAGGED] seg 005 | conf=0.55 | If you give us a small introduction...
    [⚠ FLAGGED] seg 006 | conf=0.65 | so that our audience that you are listening to tod...
    [✔] seg 007 | conf=0.82 | they know that what a brilliant...
    [✔] seg 008 | conf=0.91 | Academian you are...
    [✔] seg 009 | conf=0.93 | Thank you so much Zainab...
    [✔] seg 010 | conf=0.88 | for inviting me on your show...
    [✔] seg 011 | conf=0.87 | My name is Majda Kazmi...
    [⚠ FLAGGED] seg 012 | conf=0.74 | I am a PhD doctor...
    [⚠ FLAGGED] seg 013 | conf=0.68 | MBBS...
    [⚠ FLAGGED] seg 014 | conf=0.41 | Exactly, you ne

In [15]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Source lang    : urd_Arab (Urdu)
  Target lang    : eng_Latn (English)
  Chunk size     : 500 chars

  Loading NLLB-200 model (first run downloads ~1.2GB)...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/545 | UR: اسلام علیکم Dr. Majda...
             | EN: Islam علیکم Dr. Majda...
    ✔ seg 002/545 | UR: Welcome to the show...
             | EN: Welcome to the show...
    ✔ seg 003/545 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004/545 | UR: and taking out the time...
             | EN: today and taking the time...
    ✔ seg 005/545 | UR: If you give us a small introduction...
             | EN: if you give us a little introduction...
    ✔ seg 006/545 | UR: so that our audience that you are listen...
             | EN: so that our audience that you are listening to today...
    ✔ seg 007/545 | UR: they know that what a brilliant...
             | EN: today they know what a brilliant...
    ✔ seg 008/545 | UR: Academian you are...
             | EN: academic you are...
    ✔ seg 009/545 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    ✔ seg 010

In [16]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID   : test_audio
  Total segments : 545

  Checks: Empty translation, Translation errors, Length ratio
  NOTE: Language detection disabled (too many false positives on proper names)
    [✔] seg 001
             EN: Islam علیکم Dr. Majda...
    [✔] seg 002
             EN: Welcome to the show...
    [✔] seg 003
             EN: Thank you so much for coming today...
    [✔] seg 004
             EN: today and taking the time...
    [✔] seg 005
             EN: if you give us a little introduction...
    [✔] seg 006
             EN: so that our audience that you are listening to today...
    [✔] seg 007
             EN: today they know what a brilliant...
    [✔] seg 008
             EN: academic you are...
    [✔] seg 009
             EN: Thank you so much Zainab...
    [✔] seg 010
             EN: for inviting me to your show...
    [✔] seg 011
             EN: My name is Majda Kazmi...
    [✔] seg 012
             EN: I am a P

In [17]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...
  ✔ Presidio loaded.

  De-identifying full English text...

  De-identifying segments:
    [🔒] seg 001 | 1 entities removed | Islam علیکم Dr. [NAME]...
    [✔] seg 002 | 0 entities removed | Welcome to the show...
    [✔] seg 003 | 0 entities removed | Thank you so much for coming today...
    [🔒] seg 004 | 1 entities removed | [DATE] and taking the time...
    [✔] seg 005 | 0 entities removed | if you give us a little introduction...
    [✔] seg 006 | 0 entities removed | so that our audience that you are listening to today...
    [🔒] seg 007 | 1 entities removed | [DATE] they know what a brilliant...
    [✔] seg 008 | 0 entities removed | academic you are...
    [🔒] seg 009 | 1 entities removed | Thank you so much [NAME]...
    [✔] seg 010 | 0 entities removed | for inviting me to your show...
    [🔒] seg 011 | 1 entities removed | My name is [NAME]...
   

In [18]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')


  STAGE 6: FINAL EXPORT: JSON + DOCX
  Interview ID : test_audio

  Building final JSON dataset...
  ✔ Saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json

  Building DOCX report...
  ✔ DOCX saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ── Final Outputs ─────────────────────────────
  JSON → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
  DOCX → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ✔ Stage 6 complete. Pipeline finished!

✔ Final JSON : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
✔ Final DOCX : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx


In [19]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
import os
from google.colab import files

interview_id = stage1_result.get('interview_id', 'interview')

# Ensure we're in the repository folder where outputs should exist
print('Current working dir:', os.getcwd())

# Check if outputs folder exists
if not os.path.exists('outputs'):
    print('✗ Error: outputs/ folder not found!')
    print('   Make sure you ran Cells 6-11 (all pipeline stages) first.')
    print('\nDirectory listing:')
    os.system('ls -la')
else:
    # List what's in outputs
    print('✔ Outputs folder found. Contents:')
    os.system('ls -lh outputs/ || true')

    # Zip the entire outputs folder
    zip_base = f'{interview_id}_outputs'
    zip_file = f'{zip_base}.zip'
    try:
        shutil.make_archive(
            base_name=zip_base,
            format='zip',
            root_dir='.',
            base_dir='outputs'
        )
    except Exception as e:
        print('✗ Error creating ZIP:', e)
    else:
        if os.path.exists(zip_file):
            size_mb = os.path.getsize(zip_file) / 1e6
            print(f'\n✔ Zipped outputs: {zip_file} ({size_mb:.1f} MB)')
            print('  Downloading...')
            files.download(zip_file)
        else:
            print(f'✗ Error: Could not create {zip_file}')


Current working dir: /content/urdu-pipeline
✔ Outputs folder found. Contents:

✔ Zipped outputs: test_audio_outputs.zip (0.3 MB)
  Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

run_pipeline('audio/test_audio.mp3', start_stage=1)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  URDU INTERVIEW PROCESSING PIPELINE
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  Audio file  : audio/test_audio.mp3
  Start stage : 1
  Started at  : 2026-07-01T18:18:34.561194

  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...
  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes for long audio)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.88 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.460 | conf=0.95 | and taking out the time...
    [⚠] 00:00:07.840 → 00:00:10.820 | conf=0.55 | If you give us a small introduction...
    [⚠] 00:00:1

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/537 | UR: اسلام علیکم Dr. Majda...
             | EN: Islam علیکم Dr. Majda...
    ✔ seg 002/537 | UR: Welcome to the show...
             | EN: Welcome to the show...
    ✔ seg 003/537 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004/537 | UR: and taking out the time...
             | EN: today and taking the time...
    ✔ seg 005/537 | UR: If you give us a small introduction...
             | EN: if you give us a little introduction...
    ✔ seg 006/537 | UR: so that our audience that you are listen...
             | EN: so that our audience that you are listening to today...
    ✔ seg 007/537 | UR: they know that what a brilliant...
             | EN: today they know what a brilliant...
    ✔ seg 008/537 | UR: Academian you are...
             | EN: academic you are...
    ✔ seg 009/537 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    ✔ seg 010

    ✔ seg 373/537 | UR: your...
             | EN: your...
    ✔ seg 374/537 | UR: will...
             | EN: will...
    ✔ seg 375/537 | UR: is...
             | EN: is...
    ✔ seg 376/537 | UR: very important...
             | EN: very important...
    ✔ seg 377/537 | UR: and...
             | EN: and...


: 